# 02 — Preprocessing & Feature Engineering

**The single most important thing in this notebook:** TEP data is time-series data
grouped by `simulationRun`. Consecutive rows within the same run are highly correlated
(they're the same chemical process evolving over a few minutes). If you do a normal
random `train_test_split` on rows, you will leak information — rows from the same run
end up in both train and test, and your model will look much better than it actually is.

**The fix:** split by `simulationRun`, not by row. All rows from a given run go
entirely into train OR entirely into test, never both.


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

FAULT_COL  = "faultNumber"
RUN_COL    = "simulationRun"
SAMPLE_COL = "sample"

df = pd.read_csv("../data/processed/TEP_Random_Sample_5_percent.csv")  # same path as notebook 1
feature_cols = [c for c in df.columns if c.startswith("xmeas_") or c.startswith("xmv_")]
print("Loaded:", df.shape)


Loaded: (250000, 55)


In [5]:
# ── Split by simulationRun, stratified roughly by fault type ─────────────
# Build a run-level table: one row per run, with its fault label
run_labels = df.groupby(RUN_COL)[FAULT_COL].first().reset_index()

train_runs, test_runs = train_test_split(
    run_labels[RUN_COL],
    test_size=0.2,
    random_state=42,
    stratify=run_labels[FAULT_COL]
)

train_df = df[df[RUN_COL].isin(train_runs)].copy()
test_df  = df[df[RUN_COL].isin(test_runs)].copy()

print(f"Train: {train_df.shape[0]} rows from {train_df[RUN_COL].nunique()} runs")
print(f"Test:  {test_df.shape[0]} rows from {test_df[RUN_COL].nunique()} runs")

# sanity check: no run appears in both
overlap = set(train_df[RUN_COL]) & set(test_df[RUN_COL])
print("Overlapping runs (should be 0):", len(overlap))


Train: 199857 rows from 400 runs
Test:  50143 rows from 100 runs
Overlapping runs (should be 0): 0


## Feature Engineering

Beyond the raw 52 sensor readings, a few engineered features tend to help fault
detection models a lot, because faults often show up as *rate of change* rather
than absolute value:

- **Rolling mean / rolling std** over a short window — smooths noise, highlights drift
- **First difference** (rate of change between consecutive samples) — highlights sudden shifts
- These must be computed *within each simulationRun* — never across run boundaries.


In [10]:
def add_engineered_features(data, feature_cols, run_col, window=5):
    """Adds rolling mean, rolling std, and first-difference features per run.
    Operates within each simulationRun group so no information leaks across runs."""
    data = data.sort_values([run_col, "sample"]).copy()
    grouped = data.groupby(run_col)[feature_cols]

    roll_mean = grouped.transform(lambda s: s.rolling(window, min_periods=1).mean())
    roll_std  = grouped.transform(lambda s: s.rolling(window, min_periods=1).std().fillna(0))
    diff      = grouped.transform(lambda s: s.diff().fillna(0))

    roll_mean.columns = [f"{c}_rollmean" for c in feature_cols]
    roll_std.columns  = [f"{c}_rollstd"  for c in feature_cols]
    diff.columns      = [f"{c}_diff"     for c in feature_cols]

    return pd.concat([data, roll_mean, roll_std, diff], axis=1)

# NOTE: this multiplies your feature count by ~4x. Start WITHOUT this (use raw features
# only) to get a baseline result fast, then add this in for a second, improved result —
# that comparison itself is a great thing to show in an interview.

USE_ENGINEERED_FEATURES = True  # flip to True once you have a working baseline

if USE_ENGINEERED_FEATURES:
    train_df = add_engineered_features(train_df, feature_cols, RUN_COL)
    test_df  = add_engineered_features(test_df, feature_cols, RUN_COL)
    feature_cols = [c for c in train_df.columns if any(
        c.startswith(p) for p in ["xmeas_", "xmv_"]
    )]
    print(f"Now using {len(feature_cols)} features")


Now using 208 features


In [11]:
# ── Scale features — fit ONLY on training data ───────────────────────────
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols])
X_test  = scaler.transform(test_df[feature_cols])

y_train = train_df[FAULT_COL].values
y_test  = test_df[FAULT_COL].values
groups_train = train_df[RUN_COL].values  # needed later for GroupKFold cross-validation

print("X_train:", X_train.shape, " X_test:", X_test.shape)


X_train: (199857, 208)  X_test: (50143, 208)


In [12]:
# ── Save everything needed for the modeling notebook ────────────────────
np.save("../data/processed/X_train.npy", X_train)
np.save("../data/processed/X_test.npy", X_test)
np.save("../data/processed/y_train.npy", y_train)
np.save("../data/processed/y_test.npy", y_test)
np.save("../data/processed/groups_train.npy", groups_train)
joblib.dump(scaler, "../models/scaler.joblib")
joblib.dump(feature_cols, "../models/feature_cols.joblib")

print("Saved processed data to data/processed/ and scaler to models/")


Saved processed data to data/processed/ and scaler to models/


## Next Steps

**Next notebook:** `03_modeling_and_evaluation.ipynb` — train baseline, Random Forest,
and XGBoost classifiers, evaluate properly with grouped cross-validation, tune
hyperparameters, and save the final model.
